# Setup

In [1]:
#You might need to install these packages before running. Comment out after
!pip install torch
!pip install transformers datasets
!pip install scikit-learn
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 40.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 50.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 41.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 122.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 25.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.6/248.6 kB 33.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 9.4 MB/s eta 0:00:00


In [2]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [3]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/nlpproject/"

Mounted at /content/drive
/content/drive/My Drive/nlpproject


In [4]:
import wandb
import csv
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM,AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def multi_label_formatting(data):
    """This function does not really play a huge role for us right now, but what it basically does is that if a paragraph
    has two stances, i.e. 'conservative' and 'right', it will keep both for the classification problem.   """
    labels = []
    for i in range(len(data['stance'])):
        multi_tags = []
        if type(data['stance'][i]) is not float:
            multi_tags.append(data['stance'][i].lower())

        labels.append(multi_tags)

    return labels

#Preprocessing the CSV file that contains the BASIL database.
# df = pd.read_csv('processed_data.csv')
df = pd.read_csv('processed_data_combined.csv')

paragraphs = df["body"] # Gets paragraphs from CSV

#This line was intended to convert stances to integers. Is not needed anymore since now we use MultiLabel Binarizer for
# multilabel classification
# df['stance'] = df['stance'].replace([ 'left', 'center', 'liberal', 'conservative', 'right'],[0,1,2,3,4])


stances = df['stance'] # Gets all the stances/labels
# Stances: {'left', 'center', 'liberal', 'conservative', 'right'}
df['body'] =df['body'].astype(str)

In [6]:
df.head()

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center


In [8]:
print('dataset size:', df.shape[0])

dataset size: 37854


In [7]:
def init_data_model(batch_size, test_size):

    # Use if you would want to print a sample paragraph and label
    # print(df['body'][100])
    # print(df['stance'][100])

    #This package will convert tags to an array of size 5 (five because we have 5 stances:
    # 'left', 'center', 'liberal', 'conservative', 'right') where, for example, if a paragraph is classified as 'center'
    # it converts its label into one hot encoding [0,0,1,0,0]
    mlb = MultiLabelBinarizer()
    labels = multi_label_formatting(df) # In case of multitags. Look at function description for more info
    print(f"Labels : {labels}")
    #One Hot Enconding of Multi labels
    labels = mlb.fit_transform(labels)

    #Splitting data into test set and training set.
    x_train_og, x_test_og, y_train, y_test = train_test_split(df['body'].astype(str), labels,test_size=test_size, random_state = 0)

    #These following two models are way bigger and perform worse (tested.)
    # model_name = "roberta-large"
    # model_name = "roberta-base"

    model_name = "launch/POLITICS" # POLITICS model from HuggingFace!

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # You can check that maximum amount of tokes is 512 which means that we will not be able
    # to process the entire paragraphs.
    # print(tokenizer.model_max_length)

    # model = AutoModelForMaskedLM.from_pretrained("launch/POLITICS")
    n_labels = 3 # num_labels = 5 enables hugging face to add a classification head to the model
    model = AutoModelForSequenceClassification.from_pretrained (model_name,num_labels=n_labels)

    train_encodings = tokenizer(x_train_og.to_list(), truncation=True, padding=True, return_tensors="pt")
    train_labels = torch.tensor(y_train, dtype=torch.float32)
    train_dataset = TensorDataset(train_encodings.input_ids, train_encodings.attention_mask, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Dataloader for test data
    test_encodings = tokenizer(x_test_og.to_list(), truncation=True, padding=True, return_tensors="pt")
    test_labels = torch.tensor(y_test, dtype=torch.float32)
    test_dataset = TensorDataset(test_encodings.input_ids, test_encodings.attention_mask, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)  # No need to shuffle test data

    return train_loader, test_loader, model, tokenizer, mlb.classes_


def evaluate(test_loader, model, tokenizer, classes=None, report=False):
    # Predict on the test data
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Softmax makes more sense for single classifications
            predictions = outputs.logits.softmax(dim=-1).tolist()
            all_preds.extend(predictions)

            # In case you'd want to use Sigmoid
            # predictions = torch.sigmoid(logits)  # Apply sigmoid activation for multilabel classification
            # all_preds.extend(predictions.cpu().detach().numpy())

            all_labels.extend(labels.cpu().detach().numpy())

    # Convert the predictions and labels to binary values based on a threshold (e.g., 0.5)
    threshold = 0.5

    all_preds = (torch.tensor(all_preds) > threshold).int().numpy()
    all_labels = np.array(all_labels)

    # Compute the classification report
    accuracy = accuracy_score(all_labels, all_preds)

    # Reporting Results
    if report:
      #Bigger report summary. Sample avg is the same as Accuracy.
      report = classification_report(all_labels, all_preds, target_names=classes)
      print(report)

    # return more things want more information
    return accuracy


def train(train_loader, test_loader, model, tokenizer, lr, batch_size, threshold):
    #Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # # Fine-tuning loop
    model.to(device)

    num_epochs = 20

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            # # Backward pass and optimization
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_acc = evaluate(train_loader, model, tokenizer)
        val_acc = evaluate(test_loader, model, tokenizer)

        print(f"train_acc: {train_acc}")
        print(f"val_acc: {val_acc}")

        wandb.log({
            'loss': total_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
          })

        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss}")
        # test_model() Use if you would want to take a look at how the model is currently doing on the test data. BAD PRACTICE!
        if total_loss < threshold:
          break

    return model, tokenizer

# One-off training
This section is for if you just want to train a single model with a given configuration. Record your configuration in the following wandb config, and simply run the training block. The loss will be reported to wandb.

In [9]:
lr = 5e-5
batch_size = 16
test_size = 0.1
threshold = 2

wandb.init(
    project='politics_more_data',
    config= {
        'learning_rate': lr,
        'batch_size': batch_size,
        'test_size': test_size,
        'threshold': 2,
    }
)

wandb: Currently logged in as: ellieyhc (probgram). Use `wandb login --relogin` to force relogin


In [10]:
# Load model directly
train_loader, test_loader, model, tokenizer, classes = init_data_model(batch_size, test_size)

Labels : [['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['center'], ['center'], ['right'], ['left'], ['right'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['center'], ['center'], ['right'], ['right'], ['left'], ['center'], ['center'], ['center'], ['left'], ['right'], ['center'], ['left'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['right'], ['right'], ['left'], ['left'], ['center'], ['left'], ['center'], ['center'], ['right'], ['center'], ['left'], ['center'], ['left'], ['right'], ['left'], ['center'], ['center'], ['left'], ['left'], ['center'], ['left'], ['center'], ['left'], ['right'], ['right'], ['center'], ['center'], ['center'], ['left'], ['right'], ['right'], ['left'], ['center'], ['left'], ['center'], ['center'], ['center'], ['center'], ['center'], ['left'], ['right'], ['left'], ['left'], ['right'], ['center'], ['ri

(…)ITICS/resolve/main/tokenizer_config.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

(…)/launch/POLITICS/resolve/main/vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

(…)/launch/POLITICS/resolve/main/merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

(…)nch/POLITICS/resolve/main/tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

(…)ICS/resolve/main/special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

(…)launch/POLITICS/resolve/main/config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.bias', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model, tokenizer = train(train_loader, test_loader, model, tokenizer, lr, batch_size, threshold)

In [ ]:
# final evaluation
acc = evaluate(test_loader, model, tokenizer, classes, report=True)

              precision    recall  f1-score   support

      center       0.30      0.57      0.39        21
        left       0.36      0.20      0.26        20
       right       0.50      0.21      0.30        19

   micro avg       0.34      0.33      0.34        60
   macro avg       0.39      0.33      0.32        60
weighted avg       0.38      0.33      0.32        60
 samples avg       0.33      0.33      0.33        60



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# save the model
save_name = 'test_best'

model.save_pretrained(save_name)

In [ ]:
wandb.finish()

loss,█▇▆▅▃▂▁
train_acc,▁▂▄▇▇██
val_acc,▅▃█▄▅▅▁
loss,1.26021
train_acc,0.99167
val_acc,0.26667


# Hyperparameter Fine-tuning
This section is for doing sweeps over different hyperparameters to fine-tune for the best accuracy.

In [ ]:
sweep_config = {
    'method': 'random',
    'name': 'sweep',
    'metric': {'goal': 'maximize', 'name': 'val_acc'},
    'parameters': {
        'batch_size': {'values': [4, 8, 16]},
        'lr': {'max': 1e-4, 'min': 1e-6},
        # 'test_size': {'values': [0.2, 0.25, 0.3]},
        'threshold': {'values': [0.5, 1, 1.5, 2, 2.5, 3]}
    }
}

sweep_id = wandb.sweep(sweep=sweep_config, project='politics-sweep')

Create sweep with ID: phir15jp
Sweep URL: https://wandb.ai/probgram/politics-sweep/sweeps/phir15jp


In [ ]:
def main():
  run = wandb.init()

  lr = wandb.config.lr
  batch_size = wandb.config.batch_size
  # test_size = wandb.config.test_size
  test_size = 0.1
  threshold = wandb.config.threshold

  train_loader, test_loader, model, tokenizer, classes = init_data_model(batch_size, test_size)
  model, tokenizer = train(train_loader, test_loader, model, tokenizer, lr, batch_size, threshold)

  # del test_labels
  del model
  del tokenizer
  torch.cuda.empty_cache()


In [ ]:
wandb.agent(sweep_id, function=main, count=10)

wandb: Agent Starting Run: n7vt6bh9 with config:
wandb: 	batch_size: 16
wandb: 	lr: 3.525175838976947e-05
wandb: 	threshold: 0.5
wandb: Currently logged in as: ellieyhc (probgram). Use `wandb login --relogin` to force relogin


Labels : [['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['center'], ['center'], ['right'], ['left'], ['right'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['center'], ['center'], ['right'], ['right'], ['left'], ['center'], ['center'], ['center'], ['left'], ['right'], ['center'], ['left'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['right'], ['right'], ['left'], ['left'], ['center'], ['left'], ['center'], ['center'], ['right'], ['center'], ['left'], ['center'], ['left'], ['right'], ['left'], ['center'], ['center'], ['left'], ['left'], ['center'], ['left'], ['center'], ['left'], ['right'], ['right'], ['center'], ['center'], ['center'], ['left'], ['right'], ['right'], ['left'], ['center'], ['left'], ['center'], ['center'], ['center'], ['center'], ['center'], ['left'], ['right'], ['left'], ['left'], ['right'], ['center'], ['ri

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.bias', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


train_acc: 0.4925925925925926
val_acc: 0.36666666666666664
Epoch 1/20, Loss: 10.629566133022308
train_acc: 0.44074074074074077
val_acc: 0.3
Epoch 2/20, Loss: 10.045863509178162
train_acc: 0.5814814814814815
val_acc: 0.43333333333333335
Epoch 3/20, Loss: 9.18162938952446
train_acc: 0.6666666666666666
val_acc: 0.43333333333333335
Epoch 4/20, Loss: 8.109355926513672
train_acc: 0.9222222222222223
val_acc: 0.3333333333333333
Epoch 5/20, Loss: 6.38463032245636
train_acc: 0.9555555555555556
val_acc: 0.3333333333333333
Epoch 6/20, Loss: 4.270538121461868
train_acc: 0.9888888888888889
val_acc: 0.36666666666666664
Epoch 7/20, Loss: 2.1683145314455032
train_acc: 1.0
val_acc: 0.3
Epoch 8/20, Loss: 1.019400779157877
train_acc: 1.0
val_acc: 0.4666666666666667
Epoch 9/20, Loss: 0.6150173619389534
train_acc: 0.9888888888888889
val_acc: 0.3
Epoch 10/20, Loss: 0.5678565129637718
train_acc: 1.0
val_acc: 0.43333333333333335
Epoch 11/20, Loss: 0.8373240847140551
train_acc: 1.0
val_acc: 0.3
Epoch 12/20, Los

loss,██▇▆▅▄▂▁▁▁▁▁
train_acc,▂▁▃▄▇▇██████
val_acc,▄▁▇▇▂▂▄▁█▁▇▁
loss,0.41175
train_acc,1.0
val_acc,0.3


...

loss,██▇▇▆▆▄▃▂▁
train_acc,▁▂▁▂▃▄▇▇██
val_acc,▅█▁██████▅
loss,1.48527
train_acc,0.99259
val_acc,0.33333


wandb: Agent Starting Run: 0of0tvi1 with config:
wandb: 	batch_size: 4
wandb: 	lr: 8.543690938299851e-05
wandb: 	threshold: 2


Labels : [['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['center'], ['center'], ['right'], ['left'], ['right'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['center'], ['center'], ['right'], ['right'], ['left'], ['center'], ['center'], ['center'], ['left'], ['right'], ['center'], ['left'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['right'], ['right'], ['right'], ['left'], ['left'], ['center'], ['left'], ['center'], ['center'], ['right'], ['center'], ['left'], ['center'], ['left'], ['right'], ['left'], ['center'], ['center'], ['left'], ['left'], ['center'], ['left'], ['center'], ['left'], ['right'], ['right'], ['center'], ['center'], ['center'], ['left'], ['right'], ['right'], ['left'], ['center'], ['left'], ['center'], ['center'], ['center'], ['center'], ['center'], ['left'], ['right'], ['left'], ['left'], ['right'], ['center'], ['ri

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.bias', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


train_acc: 0.48518518518518516
val_acc: 0.36666666666666664
Epoch 1/20, Loss: 42.97725224494934
train_acc: 0.4925925925925926
val_acc: 0.36666666666666664
Epoch 2/20, Loss: 41.948015838861465
train_acc: 0.4925925925925926
val_acc: 0.36666666666666664
Epoch 3/20, Loss: 42.073275834321976


In [ ]:
wandb.finish()

# Free up memory

In [ ]:
#Memory Management
# del df
# del test_labels
del model
del tokenizer
torch.cuda.empty_cache()

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7d8212c78190>> (for pre_run_cell):


BrokenPipeError: ignored

NameError: ignored

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7d8212c78190>> (for post_run_cell):


BrokenPipeError: ignored